<a href="https://colab.research.google.com/github/nexageapps/LLM/blob/main/04_Expert/L50_Agent_Systems.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# L50: Agent Systems - LLM Agents, ReAct, and Planning

**Author:** Karthik Arjun  
**LinkedIn:** [Karthik Arjun](https://www.linkedin.com/in/karthik-arjun-a5b4a258/)  
**Level:** Expert  
**Lesson:** 50 of 60

---

## Learning Objectives

By the end of this lesson, you will:
- Build a minimal agent loop: think → act → observe → repeat
- Implement a simple ReAct-style prompt (Reasoning + Acting)
- Use a tool (e.g. calculator, search) inside the loop

---

## Concept: LLM Agents

**Agents** use an LLM to decide actions (e.g. call a tool, finish). **ReAct**: interleave reasoning and action in text (Thought / Action / Observation). **Planning**: break goal into steps; execute and re-plan. We implement a minimal loop with one tool.

---

In [ ]:
!pip install transformers torch -q
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import re
import warnings
warnings.filterwarnings('ignore')
print("Libraries loaded.")

## Part 1: Define a Simple Tool

---

In [ ]:
def calculator(expr):
    try:
        return str(eval(expr))
    except:
        return "Error"

TOOLS = {"calculator": calculator}
print("Tool: calculator(expression)")

## Part 2: Agent Loop (ReAct-Style)

---

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token
generator = pipeline("text-generation", model="gpt2", tokenizer=tokenizer)

def run_agent(query, max_steps=3):
    prompt = f"""You have a calculator. Use it when you need to compute.
Question: {query}
Thought: I need to calculate.
Action: calculator(2+3)
Observation: 5
Thought: I have the answer.
Answer: 5

Question: {query}
Thought:"""
    for step in range(max_steps):
        out = generator(prompt, max_new_tokens=80, do_sample=False, pad_token_id=tokenizer.eos_token_id)[0]["generated_text"]
        new_text = out[len(prompt):]
        prompt = out
        if "Action:" in new_text:
            match = re.search(r"calculator\s*\(([^)]+)\)", new_text)
            if match:
                obs = TOOLS["calculator"](match.group(1).strip())
                prompt += f"\nObservation: {obs}\n"
        if "Answer:" in new_text:
            break
    return prompt

result = run_agent("What is 10 * 5?")
print(result[-400:])

## Part 3: Extensions

---

In [ ]:
print("Production: use LangChain, AutoGPT, or custom loops with tool schemas and parsing.")

## Exercises

1. Add a second tool (e.g. current date) and prompt the agent to use both.
2. Parse Action: tool_name(args) robustly and handle errors.
3. Implement a simple planner: first output a list of steps, then execute each.

---

## Key Takeaways

1. Agent = loop: LLM output → parse action → run tool → add observation → repeat.
2. ReAct format (Thought/Action/Observation) improves reliability.
3. Tool schema and parsing are critical for robustness.

---

## Next Lesson

**L51: Tool Use and Function Calling** – APIs and tool integration.

---